# 数据清洗与准备

在进行数据分析和建模的过程中，大量时间都花在了数据准备上：加载、清洗、转换和重新排列。这些任务通常被报告占据了分析师时间的80%或更多。有时，数据以文件或数据库的形式存储的方式并不适合特定的任务。许多研究人员选择使用通用编程语言（如Python、Perl、R或Java）或Unix文本处理工具（如sed或awk）将数据从一种形式转换为另一种形式进行特设处理。幸运的是，pandas以及内置的Python语言特性为你提供了一套高级别、灵活且快速的工具，使你能够将数据操纵成正确的格式。

在本章中，我讨论了缺失数据、重复数据、字符串操作以及其他一些分析数据转换的工具。在下一章中，我将重点讨论以多种方式组合和重新排列数据集。

## 处理缺失数据

缺失数据在许多数据分析应用中都很常见。pandas的目标之一是尽可能无痛地处理缺失数据。例如，pandas对象上的所有描述性统计默认都排除了缺失数据。

在pandas对象中缺失数据的表示方式有些不完美，但对于大多数现实世界的使用来说已经足够了。对于具有float64数据类型的DataFrame，pandas使用浮点值NaN（非数字）来表示缺失数据。

我们称这个值为哨兵值：当它存在时，表示缺失（或空）值：

In [2]:
import pandas as pd
import numpy as np

float_data = pd.Series([1.2, -3.5, np.nan, 0])
float_data

0    1.2
1   -3.5
2    NaN
3    0.0
dtype: float64

isna方法为我们提供了一个布尔序列，其中值为null的地方为True：

In [3]:
float_data.isna()

0    False
1    False
2     True
3    False
dtype: bool

在pandas中，我们采用了R编程语言中的约定，将缺失数据称为NA，代表不可用。在统计应用中，NA数据可能是根本不存在的数据，或者是存在但未被观测到的数据（例如由于数据收集问题）。在清理数据进行分析时，通常需要对缺失数据本身进行分析，以识别数据收集问题或由缺失数据引起的潜在偏见。

Python内置的None值也被视为NA：

In [4]:
string_data = pd.Series(["aardvark", np.nan, None, "avocado"])
string_data

0    aardvark
1         NaN
2        None
3     avocado
dtype: object

In [5]:
string_data.isna()

0    False
1     True
2     True
3    False
dtype: bool

In [6]:
float_data = pd.Series([1, 2, None], dtype='float64')
float_data

0    1.0
1    2.0
2    NaN
dtype: float64

In [7]:
float_data.isna()

0    False
1    False
2     True
dtype: bool

pandas项目试图使处理缺失数据的工作在数据类型之间保持一致。像pandas.isna这样的函数抽象了许多烦人的细节。有关处理缺失数据的一些函数的列表，请参下表：

| 方法 | 描述 |
|-----|------|
| dropna | 根据每个标签的值是否缺失数据来过滤轴标签，并设置不同的阈值来容忍缺失数据的数量。|
| fillna | 使用某个值或使用插值方法（如“ffill”或“bfill”）填充缺失数据。|
| isna | 返回布尔值，指示哪些值为缺失/NA。|
| notna | isna的否定，对于非NA值返回True，对于NA值返回False。|

### 过滤缺失数据

过滤缺失数据有几种方法。虽然你总是可以选择使用pandas.isna和布尔索引手动完成，但dropna可能会很有用。在Series上，它会返回一个只包含非空数据和索引值的Series：

In [8]:
data = pd.Series([1, np.nan, 3.5, np.nan, 7])

data.dropna()

0    1.0
2    3.5
4    7.0
dtype: float64

这与做同样的事情是相同的：

In [9]:
data[data.notna()]

0    1.0
2    3.5
4    7.0
dtype: float64

对于DataFrame对象，有多种方法可以删除缺失数据。您可能希望删除所有值为NA的行或列，或者只删除包含任何NA的行或列。默认情况下，dropna会删除包含缺失值的任何行：

In [10]:
data = pd.DataFrame([[1, 6.5, 3], [1, np.nan, np.nan], [np.nan, np.nan, np.nan], [np.nan, 6.5, 3]])
data

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
2,NaN,NaN,NaN
3,NaN,6.5,3.0


In [11]:
data.dropna()

,0,1,2
0,1.0,6.5,3.0


传递how="all"将只删除所有值为NA的行：

In [12]:
data.dropna(how='all')

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
3,NaN,6.5,3.0


请记住，这些函数默认返回新对象，并不会修改原始对象的内容。

要以相同的方式删除列，请传递axis="columns"：

In [13]:
data[4] = np.nan
data

,0,1,2,4
0,1.0,6.5,3.0,NaN
1,1.0,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,NaN,6.5,3.0,NaN


In [14]:
data.dropna(axis='columns', how='all')

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
2,NaN,NaN,NaN
3,NaN,6.5,3.0


假设您只想保留最多包含一定数量缺失观测值的行。您可以使用thresh参数来指示这一点：

In [15]:
df = pd.DataFrame(np.random.standard_normal((7, 3)))
df

,0,1,2
0,-1.791709,-0.902061,-1.782957
1,-1.030418,-1.147540,-0.561689
2,-0.296509,0.916528,0.231742
3,0.977329,-1.113898,0.005866
4,0.193349,0.392732,0.858037
5,-1.734972,0.107670,-0.070406
6,0.531789,-1.247497,0.394332


In [16]:
df.iloc[:4, 1] = np.nan
df.iloc[:2, 2] = np.nan

df

,0,1,2
0,-1.791709,NaN,NaN
1,-1.030418,NaN,NaN
2,-0.296509,NaN,0.231742
3,0.977329,NaN,0.005866
4,0.193349,0.392732,0.858037
5,-1.734972,0.107670,-0.070406
6,0.531789,-1.247497,0.394332


In [17]:
df.dropna()

,0,1,2
4,0.193349,0.392732,0.858037
5,-1.734972,0.107670,-0.070406
6,0.531789,-1.247497,0.394332


In [18]:
df.dropna(thresh=2)

,0,1,2
2,-0.296509,NaN,0.231742
3,0.977329,NaN,0.005866
4,0.193349,0.392732,0.858037
5,-1.734972,0.107670,-0.070406
6,0.531789,-1.247497,0.394332


### 填补缺失数据

与其过滤掉缺失数据（并可能同时丢弃其他数据），您可能希望以多种方式填补“空缺”。对于大多数目的来说，fillna方法是首选的函数。使用常数调用fillna会将缺失值替换为该值：

In [19]:
df.fillna(0)

,0,1,2
0,-1.791709,0.000000,0.000000
1,-1.030418,0.000000,0.000000
2,-0.296509,0.000000,0.231742
3,0.977329,0.000000,0.005866
4,0.193349,0.392732,0.858037
5,-1.734972,0.107670,-0.070406
6,0.531789,-1.247497,0.394332


使用字典调用fillna方法时，可以为每个列指定不同的填充值：

In [20]:
df.fillna({1: 0.5, 2: 0})

,0,1,2
0,-1.791709,0.500000,0.000000
1,-1.030418,0.500000,0.000000
2,-0.296509,0.500000,0.231742
3,0.977329,0.500000,0.005866
4,0.193349,0.392732,0.858037
5,-1.734972,0.107670,-0.070406
6,0.531789,-1.247497,0.394332


可以使用与reindex相同的插值方法与fillna一起使用：

In [21]:
df = pd.DataFrame(np.random.standard_normal((6, 3)))
df.iloc[2:, 1] = np.nan
df.iloc[4:, 2] = np.nan
df

,0,1,2
0,2.613560,-1.919611,1.289443
1,-0.867689,1.635863,0.277015
2,0.331062,NaN,0.625680
3,-0.088993,NaN,0.667045
4,0.651972,NaN,NaN
5,-0.192609,NaN,NaN


In [22]:
df.fillna(method='ffill')

/var/folders/c5/vh03t8zn4797kc18lgrjtcbr0000gn/T/ipykernel_4990/1193302488.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill')


,0,1,2
0,2.613560,-1.919611,1.289443
1,-0.867689,1.635863,0.277015
2,0.331062,1.635863,0.625680
3,-0.088993,1.635863,0.667045
4,0.651972,1.635863,0.667045
5,-0.192609,1.635863,0.667045


In [23]:
df.fillna(method='ffill', limit=2)

/var/folders/c5/vh03t8zn4797kc18lgrjtcbr0000gn/T/ipykernel_4990/2719175769.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', limit=2)


,0,1,2
0,2.613560,-1.919611,1.289443
1,-0.867689,1.635863,0.277015
2,0.331062,1.635863,0.625680
3,-0.088993,1.635863,0.667045
4,0.651972,NaN,0.667045
5,-0.192609,NaN,0.667045


使用fillna你可以做许多其他事情，例如使用中位数或均值统计进行简单的数据插补：

In [24]:
data = pd.Series([1, np.nan, 3.5, np.nan, 7])
data.fillna(data.mean())

0    1.000000
1    3.833333
2    3.500000
3    3.833333
4    7.000000
dtype: float64

**fillna函数参数:**

| 参数 | 描述 |
|-----|------|
| value | 用于填充缺失值的标量值或类似字典的对象 |
| method | 插值方法：可以是“bfill”（向后填充）或“ffill”（向前填充）；默认值为None |
| axis | 要填充的轴（“索引”或“列”）；默认值为axis="索引" |
| limit | 对于向前和向后填充，要填充的最大连续期间数 |

## 数据转换

到目前为止，本章我们一直在关注缺失数据的处理。过滤、清洗和其他转换是另一类重要的操作。

### 删除重复项
DataFrame中可能会因为各种原因出现重复的行。这里有一个例子：


In [25]:
data = pd.DataFrame({"k1": ["one", "two"] * 3 + ["two"], "k2": [1, 1, 2, 3, 3, 4, 4]})
data

,k1,k2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4
6,two,4


DataFrame方法duplicated返回一个布尔型序列，指示每一行是否是重复的（其列值与之前某行的完全相同）：

In [26]:
data.duplicated()

0    False
1    False
2    False
3    False
4    False
5    False
6     True
dtype: bool

相关地，drop_duplicates返回一个DataFrame，其中过滤掉了重复数组为False的行：

In [27]:
data.drop_duplicates()

,k1,k2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4


默认情况下，这两种方法都会考虑所有列；或者，您可以指定它们中的任何子集来检测重复项。假设我们有一个额外的值列，并且只想基于“k1”列过滤重复项：

In [28]:
data["v1"] = range(7)
data

,k1,k2,v1
0,one,1,0
1,two,1,1
2,one,2,2
3,two,3,3
4,one,3,4
5,two,4,5
6,two,4,6


In [29]:
data.drop_duplicates(subset=["k1"])

,k1,k2,v1
0,one,1,0
1,two,1,1


默认情况下，duplicated和drop_duplicates会保留第一次观察到的值组合。传递keep="last"将返回最后一个：

In [30]:
data.drop_duplicates(["k1", "k2"], keep="last")

,k1,k2,v1
0,one,1,0
1,two,1,1
2,one,2,2
3,two,3,3
4,one,3,4
6,two,4,6


### 使用函数或映射转换数据
对于许多数据集，您可能希望根据数组、序列或DataFrame中的列的值执行一些转换。考虑以下收集的关于各种肉类数据的假设情况：

In [31]:
data = pd.DataFrame({"food": ["bacon", "pulled pork", "bacon",
                              "pastrami", "corned beef", "bacon",
                              "pastrami", "honey ham", "nova lox"],
                     "ounces": [4, 3, 12, 6, 7.5, 8, 3, 5, 6]})

data

,food,ounces
0,bacon,4.0
1,pulled pork,3.0
2,bacon,12.0
3,pastrami,6.0
4,corned beef,7.5
5,bacon,8.0
6,pastrami,3.0
7,honey ham,5.0
8,nova lox,6.0


假设你想添加一列来指示每种食物来自哪种动物。让我们写下每种不同的肉类类型到动物的种类的映射：

In [32]:
meat_to_animal = {
    "bacon": "pig",
    "pulled pork": "pig",
    "pastrami": "cow",
    "corned beef": "cow",
    "honey ham": "pig",
    "nova lox": "salmon",
}

Series上的map方法（也在第5.2.5章：函数应用与映射中讨论）接受一个函数或字典样对象，其中包含要进行值转换的映射：

In [33]:
data['animal'] = data['food'].map(meat_to_animal)
data

,food,ounces,animal
0,bacon,4.0,pig
1,pulled pork,3.0,pig
2,bacon,12.0,pig
3,pastrami,6.0,cow
4,corned beef,7.5,cow
5,bacon,8.0,pig
6,pastrami,3.0,cow
7,honey ham,5.0,pig
8,nova lox,6.0,salmon


我们也可以传递一个函数来做所有的工作：

In [34]:
def get_animal(x):
    return meat_to_animal[x]

data['food'].map(get_animal)

0       pig
1       pig
2       pig
3       cow
4       cow
5       pig
6       cow
7       pig
8    salmon
Name: food, dtype: object

使用map是执行逐元素转换和其他与数据清洗相关的操作的便捷方法。

### 替换值

使用fillna方法填充缺失数据是更一般值替换的一个特例。正如您已经看到的，map可以用来修改对象中的一部分值，但replace提供了一种更简单、更灵活的方法来完成此操作。让我们考虑这个Series：

In [35]:
data = pd.Series([1, -999, 2, -999, -1000, 3])
data

0       1
1    -999
2       2
3    -999
4   -1000
5       3
dtype: int64

-999值可能是缺失数据的哨兵值。要将其替换为pandas理解的NA值，我们可以使用replace方法，生成一个新的Series：

In [36]:
data.replace(-999, np.nan)

0       1.0
1       NaN
2       2.0
3       NaN
4   -1000.0
5       3.0
dtype: float64

如果您想一次替换多个值，您可以传递一个列表，然后是替代值：

In [37]:
data.replace([-999, -1000], np.nan)

0    1.0
1    NaN
2    2.0
3    NaN
4    NaN
5    3.0
dtype: float64

要使用不同的替换值，请传递一个替代值的列表：

In [38]:
data.replace([-999, -1000], [np.nan, 0])

0    1.0
1    NaN
2    2.0
3    NaN
4    0.0
5    3.0
dtype: float64

> data.replace方法与data.str.replace不同，后者执行逐元素字符串替换。我们将在本章后面讨论这些字符串方法在Series上的应用。

### 重命名轴索引

像Series中的值一样，轴标签可以通过某种形式的函数或映射进行类似转换，以产生新的、不同标记的对象。您还可以在不创建新数据结构的情况下就地修改轴。这里有一个简单的例子：

In [39]:
data = pd.DataFrame(np.arange(12).reshape((3, 4)), 
                    index=["Ohio", "Colorado", "New York"],
                    columns=["one", "two", "three", "four"])
data

,one,two,three,four
Ohio,0,1,2,3
Colorado,4,5,6,7
New York,8,9,10,11


像Series一样，轴索引有一个map方法：

In [40]:
def transform(x):
    return x[:4].upper()

data.index.map(transform)

Index(['OHIO', 'COLO', 'NEW '], dtype='object')

你可以将索引属性分配给DataFrame，就地修改它：

In [41]:
data.index = data.index.map(transform)
data

,one,two,three,four
OHIO,0,1,2,3
COLO,4,5,6,7
NEW,8,9,10,11


如果您想在不修改原始数据集的情况下创建一个转换版本的数据集，一种有用的方法是rename：

In [42]:
data.rename(index=str.title, columns=str.upper)

,ONE,TWO,THREE,FOUR
Ohio,0,1,2,3
Colo,4,5,6,7
New,8,9,10,11


值得注意的是，重命名可以与类似字典的对象一起使用，为轴标签的子集提供新值：

In [43]:
data.rename(index={"OHIO": "INDIANA"},
            columns={"three": "peekaboo"})

,one,two,peekaboo,four
INDIANA,0,1,2,3
COLO,4,5,6,7
NEW,8,9,10,11


rename 可以避免手动复制DataFrame并为其索引和列属性分配新值的繁琐操作。

### 离散化和分箱

连续数据通常会被离散化或以其他方式划分为“区间”进行分析。假设你有一组关于研究人群的数据，并且你想将他们分组到离散的年龄桶中：

In [44]:
ages = [20, 22, 25, 27, 21, 23, 37, 31, 61, 45, 41, 32]

让我们把这些数据分成18到25岁、26到35岁、36到60岁以及61岁及以上四个区间。为此，你需要使用pandas.cut函数：

In [45]:
bins = [18, 25, 35, 60, 100]
age_categories = pd.cut(ages, bins)
age_categories

[(18, 25], (18, 25], (18, 25], (25, 35], (18, 25], ..., (25, 35], (60, 100], (35, 60], (35, 60], (25, 35]]
Length: 12
Categories (4, interval[int64, right]): [(18, 25] < (25, 35] < (35, 60] < (60, 100]]

pandas返回的对象是一个特殊的Categorical对象。您看到的输出描述了pandas.cut计算出的区间。每个区间由一个特殊的（pandas独有的）区间值类型标识，其中包含每个区间的下限和上限：

In [46]:
age_categories.codes

array([0, 0, 0, 1, 0, 0, 2, 1, 3, 2, 2, 1], dtype=int8)

In [47]:
age_categories.categories

IntervalIndex([(18, 25], (25, 35], (35, 60], (60, 100]], dtype='interval[int64, right]')

In [48]:
age_categories.categories[0]

Interval(18, 25, closed='right')

In [49]:
pd.value_counts(age_categories)

/var/folders/c5/vh03t8zn4797kc18lgrjtcbr0000gn/T/ipykernel_4990/3010498523.py:1: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  pd.value_counts(age_categories)


(18, 25]     5
(25, 35]     3
(35, 60]     3
(60, 100]    1
Name: count, dtype: int64

请注意，pd.value_counts(categories)是pandas.cut结果的分箱计数。

在区间字符串表示中，括号意味着该侧是开放的（不包括），而方括号意味着它是封闭的（包括）。您可以通过传递right=False来更改哪一侧是封闭的：

In [50]:
pd.cut(ages, bins, right=False)

[[18, 25), [18, 25), [25, 35), [25, 35), [18, 25), ..., [25, 35), [60, 100), [35, 60), [35, 60), [25, 35)]
Length: 12
Categories (4, interval[int64, left]): [[18, 25) < [25, 35) < [35, 60) < [60, 100)]

您可以通过向labels选项传递一个列表或数组来覆盖默认的基于间隔的分箱标签：

In [51]:
group_names = ["Youth", "YoungAdult", "MiddleAged", "Senior"]
pd.cut(ages, bins, labels=group_names)

['Youth', 'Youth', 'Youth', 'YoungAdult', 'Youth', ..., 'YoungAdult', 'Senior', 'MiddleAged', 'MiddleAged', 'YoungAdult']
Length: 12
Categories (4, object): ['Youth' < 'YoungAdult' < 'MiddleAged' < 'Senior']

如果你传递给pandas.cut的是一个整数数量的区间而不是显式的区间边界，它将基于数据中的最小值和最大值计算等长度的区间。考虑一些均匀分布的数据被切成四份的情况：

In [52]:
data = np.random.uniform(size=20)
pd.cut(data, 4, precision=2)

[(0.7, 0.92], (0.7, 0.92], (0.26, 0.48], (0.48, 0.7], (0.043, 0.26], ..., (0.26, 0.48], (0.48, 0.7], (0.26, 0.48], (0.26, 0.48], (0.48, 0.7]]
Length: 20
Categories (4, interval[float64, right]): [(0.043, 0.26] < (0.26, 0.48] < (0.48, 0.7] < (0.7, 0.92]]

precision=2选项将小数精度限制为两位。

一个与之密切相关的函数是pandas.qcut，它根据样本分位数对数据进行分箱。根据数据的分布情况，使用pandas.cut通常不会导致每个箱子中的数据点数量相同。由于pandas.qcut使用的是样本分位数而不是数据点的中位数或均值，因此你将得到大致相等大小的箱子：

In [53]:
data = np.random.standard_normal(1000)
quartiles = pd.qcut(data, 4, precision=2)
quartiles

[(-2.82, -0.7], (0.011, 0.67], (-0.7, 0.011], (-2.82, -0.7], (-0.7, 0.011], ..., (0.67, 3.54], (0.67, 3.54], (0.011, 0.67], (-0.7, 0.011], (-0.7, 0.011]]
Length: 1000
Categories (4, interval[float64, right]): [(-2.82, -0.7] < (-0.7, 0.011] < (0.011, 0.67] < (0.67, 3.54]]

In [54]:
pd.value_counts(quartiles)

/var/folders/c5/vh03t8zn4797kc18lgrjtcbr0000gn/T/ipykernel_4990/3472704981.py:1: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  pd.value_counts(quartiles)


(-2.82, -0.7]    250
(-0.7, 0.011]    250
(0.011, 0.67]    250
(0.67, 3.54]     250
Name: count, dtype: int64

与pandas.cut类似，您可以传递您自己的分位数（介于0和1之间的数字，包括0和1）：

In [55]:
pd.qcut(data, [0, 0.1, 0.5, 0.9, 1]).value_counts()

(-2.814, -1.276]    100
(-1.276, 0.0109]    400
(0.0109, 1.207]     400
(1.207, 3.541]      100
Name: count, dtype: int64

我们将在本章后面讨论聚合和分组操作时回到pandas.cut和pandas.qcut，因为这些离散化函数对于分位数和分组分析特别有用。

### 异常检测与过滤
过滤或转换异常值在很大程度上是应用数组操作的问题。考虑一个包含一些正态分布数据的DataFrame：

In [56]:
data = pd.DataFrame(np.random.standard_normal((1000, 4)))
data.describe()

,0,1,2,3
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,-0.046093,0.011021,0.024371,-0.000074
std,1.015622,1.000189,1.009700,0.975381
min,-3.803486,-3.621597,-3.225861,-3.101332
25%,-0.708856,-0.675757,-0.652550,-0.656077
50%,-0.071851,0.015029,0.027729,-0.018299
75%,0.616933,0.706939,0.762467,0.658434
max,3.321078,3.018218,3.297271,3.561241


假设你想找某一列中绝对值大于3的值：

In [57]:
col = data[2]
col

0     -1.676200
1     -0.261380
2     -1.615529
3      0.606692
4     -2.341080
         ...   
995   -0.147213
996    1.181618
997    0.107158
998   -0.972405
999    0.132604
Name: 2, Length: 1000, dtype: float64

In [58]:
col[col.abs() > 3]

82     3.013894
247   -3.225861
538    3.161750
761    3.297271
770    3.216733
Name: 2, dtype: float64

要选择所有值超过3或-3的行，您可以使用布尔数据框上的任何方法：

In [59]:
data[(data.abs() > 3).any(axis="columns")]

,0,1,2,3
38,0.518914,3.018218,0.400194,0.111502
82,-0.835867,1.685720,3.013894,-0.620899
174,3.321078,0.955284,-1.528811,-2.762072
197,0.972945,-3.120420,-2.341051,0.298040
223,-3.252293,-0.692755,-1.150756,0.993052
247,0.728031,-1.528386,-3.225861,-0.338715
420,1.262723,-1.367839,-0.951600,3.561241
458,-0.510252,-0.313771,-1.048305,-3.101332
538,-0.340679,1.333962,3.161750,1.257634
661,-3.443292,0.890749,-1.118984,1.097903


data.abs() > 3两边的括号是必要的，以便调用比较操作结果上的任何方法。

可以根据这些标准设置值。以下是限制超出-3到3区间值的代码：

In [60]:
data[data.abs()>3] = np.sign(data) * 3
data.describe()

,0,1,2,3
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,-0.044864,0.011745,0.023907,-0.000680
std,1.009628,0.997694,1.006879,0.972709
min,-3.000000,-3.000000,-3.000000,-3.000000
25%,-0.708856,-0.675757,-0.652550,-0.656077
50%,-0.071851,0.015029,0.027729,-0.018299
75%,0.616933,0.706939,0.762467,0.658434
max,3.000000,3.000000,3.000000,3.000000


语句np.sign(data)根据数据中的值是正数还是负数产生1和-1的值：

In [61]:
np.sign(data).head()

,0,1,2,3
0,-1.0,-1.0,-1.0,-1.0
1,1.0,-1.0,-1.0,-1.0
2,-1.0,-1.0,-1.0,-1.0
3,1.0,-1.0,1.0,1.0
4,1.0,-1.0,-1.0,-1.0


### 排列与随机抽样

可以使用numpy.random.permutation函数对Series或DataFrame中的行进行随机重排序。调用permutation时传入你想要重排的轴的长度，将产生一个整数数组，指示新的排序顺序：

In [62]:
df = pd.DataFrame(np.arange(5*7).reshape((5, 7)))
df

,0,1,2,3,4,5,6
0,0,1,2,3,4,5,6
1,7,8,9,10,11,12,13
2,14,15,16,17,18,19,20
3,21,22,23,24,25,26,27
4,28,29,30,31,32,33,34


In [63]:
sampler = np.random.permutation(5)
sampler

array([0, 2, 4, 3, 1])

然后可以使用该数组进行iloc索引或等效的take函数：

In [64]:
df.take(sampler)

,0,1,2,3,4,5,6
0,0,1,2,3,4,5,6
2,14,15,16,17,18,19,20
4,28,29,30,31,32,33,34
3,21,22,23,24,25,26,27
1,7,8,9,10,11,12,13


In [65]:
df.iloc[sampler]

,0,1,2,3,4,5,6
0,0,1,2,3,4,5,6
2,14,15,16,17,18,19,20
4,28,29,30,31,32,33,34
3,21,22,23,24,25,26,27
1,7,8,9,10,11,12,13


通过调用带有axis="columns"的take函数，我们也可以选择列的一个排列：

In [66]:
column_sampler = np.random.permutation(7)
column_sampler

array([3, 4, 2, 1, 6, 0, 5])

In [67]:
df.take(column_sampler, axis="columns")

,3,4,2,1,6,0,5
0,3,4,2,1,6,0,5
1,10,11,9,8,13,7,12
2,17,18,16,15,20,14,19
3,24,25,23,22,27,21,26
4,31,32,30,29,34,28,33


要选择一个不重复的随机子集（同一行不能出现两次），可以在Series和DataFrame上使用sample方法：

In [68]:
df.sample(n=3)

,0,1,2,3,4,5,6
4,28,29,30,31,32,33,34
1,7,8,9,10,11,12,13
0,0,1,2,3,4,5,6


要生成一个可重复选择的样本（允许重复选择），请向sample传递replace=True：

In [69]:
choices = pd.Series([5, 7, -1, 6, 4])
choices.sample(n=10, replace=True)

4    4
3    6
0    5
0    5
2   -1
0    5
0    5
4    4
2   -1
3    6
dtype: int64

### 计算指标/虚拟变量

另一种用于统计建模或机器学习应用的转换是将分类变量转换为虚拟变量或指示矩阵。如果DataFrame中的一列有k个不同的值，您将得到一个包含k列的矩阵或DataFrame，这些列全部由1和0组成。pandas有一个pandas.get_dummies函数用于执行此操作，尽管您也可以自己设计一个。让我们考虑一个示例DataFrame：

In [70]:
df = pd.DataFrame({"key": ["b", "b", "a", "c", "a", "b"], "data1": range(6)})
df

,key,data1
0,b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,b,5


In [71]:
pd.get_dummies(df["key"], dtype=float)

,a,b,c
0,0.0,1.0,0.0
1,0.0,1.0,0.0
2,1.0,0.0,0.0
3,0.0,0.0,1.0
4,1.0,0.0,0.0
5,0.0,1.0,0.0


在这里我传递了dtype=float来改变输出类型从布尔值（在较新版本的pandas中默认为布尔值）到浮点数。
在某些情况下，您可能希望在指示器DataFrame的列前添加一个前缀，然后将其与其他数据合并。pandas.get_dummies有一个prefix参数用于此操作：

In [72]:
dummies = pd.get_dummies(df["key"], prefix="key", dtype=float)
dummies

,key_a,key_b,key_c
0,0.0,1.0,0.0
1,0.0,1.0,0.0
2,1.0,0.0,0.0
3,0.0,0.0,1.0
4,1.0,0.0,0.0
5,0.0,1.0,0.0


In [73]:
df_with_dummies = df[['data1']].join(dummies)
df_with_dummies

,data1,key_a,key_b,key_c
0,0,0.0,1.0,0.0
1,1,0.0,1.0,0.0
2,2,1.0,0.0,0.0
3,3,0.0,0.0,1.0
4,4,1.0,0.0,0.0
5,5,0.0,1.0,0.0


DataFrame.join方法将在下一章中更详细地讲解。

如果数据框中的一行属于多个类别，我们必须使用不同的方法来创建虚拟变量。让我们来看看MovieLens 1M数据集，在第13章：数据分析示例中更详细地研究了它：

In [74]:
mnames = ["movie_id", "title", "genres"]
movies = pd.read_table("datasets/movielens/movies.dat", sep="::", 
                       header=None, names=mnames, engine='python')
movies.head()

,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


pandas实现了一个特殊的方法str.get_dummies（以str开头的函数将在后面的字符串操作中详细讨论），它处理了多个组成员身份编码为一个分隔符字符串的情况：

In [76]:
dummies = movies["genres"].str.get_dummies(sep="|")
dummies.head()

,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0
2,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0


然后，像之前一样，你可以使用add_prefix方法将“Genre_”添加到dummy DataFrame的列名中，同时结合电影：

In [77]:
movies_windic = movies.join(dummies.add_prefix("Genre_"))
movies_windic.head()

,movie_id,title,genres,Genre_Action,Genre_Adventure,Genre_Animation,Genre_Children's,Genre_Comedy,Genre_Crime,Genre_Documentary,...,Genre_Fantasy,Genre_Film-Noir,Genre_Horror,Genre_Musical,Genre_Mystery,Genre_Romance,Genre_Sci-Fi,Genre_Thriller,Genre_War,Genre_Western
0,1,Toy Story (1995),Animation|Children's|Comedy,0,0,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),Adventure|Children's|Fantasy,0,1,0,1,0,0,0,...,1,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),Comedy|Romance,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),Comedy|Drama,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Father of the Bride Part II (1995),Comedy,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


> 对于更大的数据集，使用多成员构建指标变量的方法并不特别快速。更好的做法是编写一个直接写入NumPy数组的低级函数，然后将结果包装在DataFrame中。

统计应用的一个有用方法是结合pandas.get_dummies和离散化函数如pandas.cut：

In [80]:
np.random.seed(12345)
values = np.random.uniform(size=10)
values

array([0.92961609, 0.31637555, 0.18391881, 0.20456028, 0.56772503,
       0.5955447 , 0.96451452, 0.6531771 , 0.74890664, 0.65356987])

In [82]:
bins = [0, 0.2, 0.4, 0.6, 0.8, 1]
cuts = pd.cut(values, bins)
cuts

[(0.8, 1.0], (0.2, 0.4], (0.0, 0.2], (0.2, 0.4], (0.4, 0.6], (0.4, 0.6], (0.8, 1.0], (0.6, 0.8], (0.6, 0.8], (0.6, 0.8]]
Categories (5, interval[float64, right]): [(0.0, 0.2] < (0.2, 0.4] < (0.4, 0.6] < (0.6, 0.8] < (0.8, 1.0]]

In [85]:
pd.get_dummies(cuts)

,"(0.0, 0.2]","(0.2, 0.4]","(0.4, 0.6]","(0.6, 0.8]","(0.8, 1.0]"
0,False,False,False,False,True
1,False,True,False,False,False
2,True,False,False,False,False
3,False,True,False,False,False
4,False,False,True,False,False
5,False,False,True,False,False
6,False,False,False,False,True
7,False,False,False,True,False
8,False,False,False,True,False
9,False,False,False,True,False


## 扩展数据类型

> 这是一个较新且更高级的话题，许多pandas用户不需要了解太多，但为了完整性，我在这里介绍它，因为我在接下来的章节中会引用和使用扩展数据类型。

pandas最初是基于NumPy的能力构建的，NumPy是一个主要用于处理数值数据的数组计算库。许多pandas概念，如缺失数据，都是使用NumPy中可用的功能实现的，同时试图在使用NumPy和pandas的库之间实现最大兼容性。

基于NumPy构建导致了一些缺点，例如：

- 对于某些数值数据类型（如整数和布尔值）的缺失数据处理不完整。因此，当缺失数据被引入这些数据时，pandas会将数据类型转换为float64并使用np.nan来表示空值。这会在许多pandas算法中引入微妙的问题，产生累积效应。
- 包含大量字符串数据的数据集在计算上代价高昂，并且占用大量的内存。
- 某些数据类型，如时间间隔、时间差和带时区的时间戳，如果不使用计算成本高昂的Python对象数组，将无法有效支持。

最近，pandas开发了一个扩展类型系统，允许添加新的数据类型，即使它们不是NumPy原生支持的。这些新数据类型可以与来自NumPy数组的数据一样作为一等对象处理。

让我们来看一个例子，我们创建了一个包含缺失值的整数序列：

In [86]:
s = pd.Series([1, 2, 3, None])
s

0    1.0
1    2.0
2    3.0
3    NaN
dtype: float64

In [87]:
s.dtype

dtype('float64')

主要是出于向后兼容的原因，Series 使用了使用 float64 数据类型和 np.nan 作为缺失值的遗留行为。我们可以使用 pandas.Int64Dtype 创建这个 Series 而不是这样：

In [88]:
s = pd.Series([1, 2, 3, None], dtype=pd.Int64Dtype())
s

0       1
1       2
2       3
3    <NA>
dtype: Int64

In [89]:
s.isna()

0    False
1    False
2    False
3     True
dtype: bool

In [90]:
s.dtype

Int64Dtype()

输出<NA>表示扩展类型数组缺少一个值。这使用了特殊的pandas.NA哨兵值：

In [91]:
s[3] is pd.NA

True

我们也可以使用简写“Int64”而不是pd.Int64Dtype()来指定类型。必须大写字母，否则它将成为基于NumPy的非扩展类型：

In [92]:
s = pd.Series([1, 2, 3, None], dtype='Int64')

pandas还有一个专门用于字符串数据的扩展类型，它不使用NumPy对象数组（需要安装pyarrow库）：

In [93]:
s = pd.Series(['one', 'two', None, 'three'], dtype=pd.StringDtype())
s

0      one
1      two
2     <NA>
3    three
dtype: string

这些字符串数组通常使用更少的内存，并且在处理大型数据集时计算效率更高。

扩展类型可以传递给Series astype方法，允许您轻松转换作为数据清洗过程的一部分：

In [94]:
df = pd.DataFrame({"A": [1, 2, None, 4],
                   "B": ['one', 'two', 'three', None],
                   "C": [False, None, False, True]})
df

,A,B,C
0,1.0,one,False
1,2.0,two,None
2,NaN,three,False
3,4.0,None,True


In [95]:
df['A'] = df['A'].astype('Int64')
df['B'] = df['B'].astype('string')
df['C'] = df['C'].astype('boolean')

df

,A,B,C
0,1,one,False
1,2,two,<NA>
2,<NA>,three,False
3,4,<NA>,True


**pandas扩展数据类型:**

| 扩展类型 | 描述 |
|---------|-----|
| BooleanDtype | 可空布尔数据，作为字符串传递时使用“boolean” |
| CategoricalDtype | 分类数据类型，作为字符串传递时使用“category” |
| DatetimeTZDtype | 带时区的日期时间 |
| Float32Dtype | 32位可空浮点型，以字符串形式传递时使用“Float32” |
| Float64Dtype | 64位可空浮点型，以字符串形式传递时使用“Float64” |
| Int8Dtype | 8位可空有符号整数，以字符串形式传递时使用“Int8” |
| Int16Dtype | 16位可空有符号整数，以字符串形式传递时使用“Int16” |
| Int32Dtype | 32位可空有符号整数，以字符串形式传递时使用“Int32” |
| Int64Dtype | 64位可空有符号整数，以字符串形式传递时使用“Int64” |
| UInt8Dtype | 8位可空无符号整数，以字符串形式传递时使用“UInt8” |
| UInt16Dtype | 16位可空无符号整数，以字符串形式传递时使用“UInt16” |
| UInt32Dtype | 32位可空无符号整数，以字符串形式传递时使用“UInt32” |
| UInt64Dtype | 64位可空无符号整数，以字符串形式传递时使用"UInt64" |

## 字符串操作

Python长期以来一直是一种流行的原始数据操作语言，部分原因在于它易于使用字符串和文本处理。大多数文本操作都可以通过字符串对象的内置方法来简化。对于更复杂的模式匹配和文本操作，可能需要正则表达式。pandas通过使您能够简洁地对整个数据数组应用字符串和正则表达式，并额外处理缺失数据的烦恼，从而增加了这一组合。

### Python内置字符串对象方法

在许多字符串处理和脚本编写应用中，内置的字符串方法就足够了。例如，一个逗号分隔的字符串可以用split方法分割成多个部分：

In [96]:
val = "a,b, guido"
val.split(",")

['a', 'b', ' guido']

split通常与strip结合使用来删除空白字符（包括换行符）：

In [97]:
pieces = [x.strip() for x in val.split(",")]
pieces

['a', 'b', 'guido']

这些子字符串可以用加号连接起来，中间用双冒号分隔符隔开：

In [98]:
first, second, third = pieces
first + "::" + second + "::" + third

'a::b::guido'

但这并不是一个实用的通用方法。一个更快、更Pythonic的方法是将列表或元组传递给字符串的join方法，使用双冒号":"：

In [99]:
"::".join(pieces)

'a::b::guido'

其他方法关注于定位子字符串。使用Python的in关键字是检测子字符串的最佳方式，尽管index和find也可以使用：

In [100]:
"guido" in val

True

In [101]:
val.index(",")

1

In [102]:
val.find(":")

-1

请注意，find和index之间的区别在于，如果字符串没有找到，index会引发异常（而不是返回-1）：

In [103]:
val.index(':')

ValueError: substring not found

相关地，count 返回特定子字符串出现的次数：

In [104]:
val.count(",")

2

replace将一个模式的所有出现替换成另一个。它通常也用来删除模式，通过传递一个空字符串：

In [105]:
val.replace(",", ":::")

'a:::b::: guido'

In [106]:
val.replace(",", "")

'ab guido'

**Python内置字符串方法:**

| 方法 | 描述 |
|------|-----|
| count | 返回子串在字符串中非重叠出现的次数 |
| endwith | 如果字符串以某个后缀结尾，则返回True |
| startwith | 如果字符串以前缀开头则返回True |
| join | 使用字符串作为分隔符来连接一系列其他字符串 |
| index | 如果字符串中找到了传入的子串，则返回该子串首次出现的位置索引；否则，如果没有找到，则引发ValueError |
| find | 返回字符串中第一次出现子串的第一个字符的索引位置；和索引一样，如果没有找到则返回 -1 |
| rfind | 返回字符串中最后一次出现子串的第一个字符的位置；如果未找到则返回-1 |
| replace | 用另一个字符串替换字符串中的出现。|
| strip, rstrip, lstrip | 分别删除左右两侧、右侧或左侧的空白字符（包括换行符）。|
| split | 使用传入的分隔符将字符串分解成子字符串列表 |
| lower | 将字母字符转换为小写 |
| upper | 将字母字符转换为大写 |
| casefold | 将字符转换为小写，并将任何特定于地区的变量字符组合转换为通用的可比形式。 |
| ljust, rjust | 左对齐或右对齐；用空格（或其他填充字符）填充字符串的另一侧以返回具有最小宽度的字符串。|

### 正则表达式

正则表达式提供了一种灵活的方法来搜索或匹配（通常更复杂的）字符串模式。单个表达式，通常称为正则表达式，是根据正则表达式语言形成的字符串。Python的内置re模块负责将正则表达式应用于字符串；我将在这里给出一些使用它的例子。

re 模块的功能分为三类：模式匹配、替换和分割。这些功能自然是相关的；正则表达式描述了一个在文本中定位的模式，然后可以用于许多目的。让我们看一个简单的例子：假设我们想要分割一个字符串，其中包含可变数量的空格字符（制表符、空格和换行符）。

描述一个或多个空白字符的正则表达式是\s+：

In [107]:
import re

text = "foo    bar\t baz  \tqux"
re.split(r'\s+', text)

['foo', 'bar', 'baz', 'qux']

当你调用re.split(r"\s+", text)时，正则表达式首先被编译，然后对其传入的文本调用split方法。你可以使用re.compile来自己编译正则表达式，形成一个可重用的正则表达式对象：

In [108]:
regex = re.compile(r'\s+')
regex.split(text)

['foo', 'bar', 'baz', 'qux']

如果你想要得到所有匹配正则表达式的模式列表，你可以使用findall方法：

In [109]:
regex.findall(text)

['    ', '\t ', '  \t']

> 为了避免在正则表达式中意外地使用反斜杠\，请使用原始字符串字面量，如r"C:\x"，而不是等效的"C:\\\\x"。

如果你打算将同一个表达式应用于许多字符串，强烈建议使用re.compile创建正则表达式对象；这样做可以节省CPU周期。

匹配和搜索与findall密切相关。虽然findall返回字符串中的所有匹配项，但search只返回第一个匹配项。更严格地说，match只匹配字符串的开头。作为一个不那么琐碎的例子，让我们考虑一段文本和一个能够识别大多数电子邮件地址的正则表达式：

In [110]:
text = """Dave dave@google.com
Steve steve@gmail.com
Rob rob@gmail.com
Ryan ryan@yahoo.com"""

pattern = r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,4}'

regex = re.compile(pattern, flags = re.IGNORECASE)

在文本中使用findall产生一个电子邮件地址列表：

In [111]:
regex.findall(text)

['dave@google.com', 'steve@gmail.com', 'rob@gmail.com', 'ryan@yahoo.com']

search返回了文本中第一个电子邮件地址的特殊匹配对象。对于前面的正则表达式，匹配对象只能告诉我们模式在字符串中的开始和结束位置：

In [112]:
m = regex.search(text)
m

<re.Match object; span=(5, 20), match='dave@google.com'>

In [113]:
text[m.start():m.end()]

'dave@google.com'

regex.match返回None，因为它只有在模式出现在字符串的开头时才会匹配：

In [114]:
print(regex.match(text))

None


相关地，sub将返回一个新字符串，其中模式的出现被新字符串替换：

In [116]:
print(regex.sub('REDACTED', text))

Dave REDACTED
Steve REDACTED
Rob REDACTED
Ryan REDACTED


假设你想找到电子邮件地址，并同时将每个地址分割成三个部分：用户名、域名和域后缀。为此，请在要分割的模式的各个部分周围加上括号：

In [119]:
pattern = r'([A-Za-z0-9._%+-]+)@([A-Za-z0-9.-]+)\.([A-Za-z]{2,4})'
regex = re.compile(pattern, flags=re.IGNORECASE)

这个修改过的正则表达式产生的匹配对象通过其groups方法返回一个包含模式组件的元组：

In [120]:
m = regex.match("wesm@bright.net")
m.groups()

('wesm', 'bright', 'net')

findall返回一个元组列表，当模式有分组时：

In [121]:
regex.findall(text)

[('dave', 'google', 'com'),
 ('steve', 'gmail', 'com'),
 ('rob', 'gmail', 'com'),
 ('ryan', 'yahoo', 'com')]

sub还可以使用特殊符号（如\1和\2）访问每场比赛中的组。符号\1对应第一个匹配的组，\2对应第二个组，依此类推：

In [122]:
print(regex.sub(r"Username:\1, Domain:\2, Suffix:\3", text))

Dave Username:dave, Domain:google, Suffix:com
Steve Username:steve, Domain:gmail, Suffix:com
Rob Username:rob, Domain:gmail, Suffix:com
Ryan Username:ryan, Domain:yahoo, Suffix:com


**正则表达式方法:**

| 方法 | 描述 |
|------|-----|
| findall | 返回字符串中所有不重叠的匹配模式作为一个列表 |
| finditer | 类似于findall，但返回一个迭代器 |
| match | 在字符串开头匹配模式，并可选地将模式组件分割成组；如果模式匹配，则返回一个匹配对象，否则返回None |
| search | 扫描字符串以匹配模式，如果匹配成功则返回一个匹配对象；与match不同，匹配可以出现在字符串的任何位置，而不仅仅是开头。|
| split | 在模式每次出现的地方将字符串分割成片段 |
| sub, subn | 将字符串中所有（sub）或前n个（subn）模式替换为替换表达式；使用符号\1、\2等来引用替换字符串中的匹配组元素 |

### pandas中的字符串函数

清理一个混乱的数据集进行分析通常需要大量的字符串操作。更糟糕的是，包含字符串的列有时会有缺失数据：

In [126]:
data = {"Dave": "dave@google.com", "Steve": "steve@gmail.com", 
        "Rob": "rob@gmail.com", "Wes": np.nan}
data = pd.Series(data)
data

Dave     dave@google.com
Steve    steve@gmail.com
Rob        rob@gmail.com
Wes                  NaN
dtype: object

In [127]:
data.isna()

Dave     False
Steve    False
Rob      False
Wes       True
dtype: bool

可以使用data.map将字符串和正则表达式方法应用于每个值（传递一个lambda或其他函数），但它会在NA（空）值上失败。为了解决这个问题，Series提供了面向数组的字符串操作方法，这些方法会跳过并传播NA值。这些方法可以通过Series的str属性访问；例如，我们可以使用str.contains检查每个电子邮件地址是否包含"gmail"：

In [128]:
data.str.contains("gmail")

Dave     False
Steve     True
Rob       True
Wes        NaN
dtype: object

请注意，此操作的结果具有object数据类型。pandas提供了扩展类型，用于专门处理字符串、整数和布尔数据，这些数据在处理缺失数据时直到最近才有一些粗糙的边缘：

In [129]:
data_as_string_text = data.astype('string')
data_as_string_text

Dave     dave@google.com
Steve    steve@gmail.com
Rob        rob@gmail.com
Wes                 <NA>
dtype: string

In [130]:
data_as_string_text.str.contains("gmail")

Dave     False
Steve     True
Rob       True
Wes       <NA>
dtype: boolean

也可以使用正则表达式，以及任何re选项，如IGNORECASE：

In [132]:
pattern = r'([A-Za-z0-9._%+-]+)@([A-Za-z0-9.-]+)\.([A-Za-z]{2,4})'
data.str.findall(pattern, flags=re.IGNORECASE)

Dave     [(dave, google, com)]
Steve    [(steve, gmail, com)]
Rob        [(rob, gmail, com)]
Wes                        NaN
dtype: object

有两种方法可以实现向量化的元素检索。要么使用str.get()，要么索引到str属性中：

In [135]:
matches = data.str.findall(pattern, flags=re.IGNORECASE).str[0]
matches

Dave     (dave, google, com)
Steve    (steve, gmail, com)
Rob        (rob, gmail, com)
Wes                      NaN
dtype: object

In [136]:
matches.str.get(1)

Dave     google
Steve     gmail
Rob       gmail
Wes         NaN
dtype: object

你可以使用类似的语法来切片字符串：

In [137]:
data.str[:5]

Dave     dave@
Steve    steve
Rob      rob@g
Wes        NaN
dtype: object

str.extract方法将以DataFrame的形式返回正则表达式的捕获组：

In [138]:
data.str.extract(pattern, flags=re.IGNORECASE)

,0,1,2
Dave,dave,google,com
Steve,steve,gmail,com
Rob,rob,gmail,com
Wes,NaN,NaN,NaN


**Series字符串方法的部分列表:**

| 方法 | 描述 |
|------|-----|
| cat | 以可选的分隔符逐个元素连接字符串 |
| contains | 如果每个字符串都包含模式/正则表达式，则返回布尔数组 |
| count | 计算模式的出现次数 |
| extract | 使用带有组的正则表达式从一系列字符串中提取一个或多个字符串；结果将是一个DataFrame，每个组对应一列 |
| endswith | 相当于对每个元素调用x.endswith(pattern) |
| startswith | 相当于对每个元素调用x.startswith(pattern) |
| findall | 计算每个字符串中模式/正则表达式的所有出现情况 |
| get | 索引到每个元素（检索第i个元素） |
| isalnum | 等同于内置的str.alnum |
| isalpha | 等同于内置的str.isalpha |
| isdecimal | 等同于内置的str.isdecimal |
| isdigit | 等同于内置的 str.isdigit |
| islower | 等同于内置的str.islower |
| isnumeric | 等同于内置的str.isnumeric |
| isupper | 等同于内置的str.isupper |
| join | 使用传递的分隔符将Series中每个元素中的字符串连接起来 |
| len | 计算每个字符串的长度 |
| lower, upper | 转换大小写；等同于对每个元素执行x.lower()或x.upper() |
| match | 使用re.match对每个元素应用传递的正则表达式，返回True或False表示是否匹配 |
| pad | 在字符串的左侧、右侧或两侧添加空格 |
| center | 等同于pad(side="both") |
| repeat | 重复值（例如，s.str.repeat(3) 对每个字符串相当于 x * 3） |
| replace | 将模式/正则表达式替换为其他字符串 |
| slice | 将Series中的每个字符串切片 |
| split | 按分隔符或正则表达式拆分字符串 |
| strip | 从两侧去除空格和换行符 |
| rstrip | 删除右侧的空格 |
| lstrip | 删除左侧的空白 |

## 分类数据

本节介绍pandas的Categorical类型。我将展示如何使用它来提高某些pandas操作的性能和内存使用效率。我还介绍了一些可能有助于在统计和机器学习应用中使用分类数据的工具。

### 背景与动机

通常情况下，表中的一列可能包含一组较小不同值的重复实例。我们已经看到了像unique和value_counts这样的函数，它们分别允许我们从数组中提取不同值并计算它们的频率：

In [139]:
values = pd.Series(['apple', 'orange', 'apple', 'apple'] * 2)
values

0     apple
1    orange
2     apple
3     apple
4     apple
5    orange
6     apple
7     apple
dtype: object

In [140]:
pd.unique(values)

array(['apple', 'orange'], dtype=object)

In [142]:
values.value_counts()

apple     6
orange    2
Name: count, dtype: int64

许多数据系统（用于数据仓库、统计计算或其他用途）已经开发了专门的方法来表示具有重复值的数据，以便更有效地存储和计算。在数据仓库中，一个最佳实践是使用所谓的维度表来包含不同的值，并将主要观测值作为整数键存储，这些键引用维度表：

In [143]:
values = pd.Series([0, 1, 0, 0] * 2)
values

0    0
1    1
2    0
3    0
4    0
5    1
6    0
7    0
dtype: int64

In [144]:
dim = pd.Series(['apple', 'orange'])

我们可以使用take方法来恢复原始字符串序列：

In [145]:
dim.take(values)

0     apple
1    orange
0     apple
0     apple
0     apple
1    orange
0     apple
0     apple
dtype: object

这种表示方法称为整数编码或字典编码。不同值的数组可以称为数据的类别、字典或级别。在本书中我们将使用术语“类别”。引用类别的整数值称为类别代码或简称代码。

在进行数据分析时，分类表示可以带来显著的性能提升。你还可以在保持代码不变的情况下对分类进行转换。一些可以相对低成本进行的示例转换包括：
- 重命名类别
- 在不改变现有类别顺序或位置的情况下添加新类别

### pandas中的分类扩展类型

pandas有一个特殊的Categorical扩展类型，用于存储使用基于整数的分类表示或编码的数据。这是一种针对具有许多相似值出现的数据的流行数据压缩技术，并且可以提供显著更快的性能以及更低的内存使用，特别是对于字符串数据。

让我们考虑之前的示例序列：

In [146]:
fruits = ['apple', 'orange', 'apple', 'apple'] * 2
N = len(fruits)
rng = np.random.default_rng(seed=12345)

df = pd.DataFrame({'fruit': fruits,
                   'basket_id': np.arange(N),
                   'count': rng.integers(3, 15, size=N),
                   'weight': rng.uniform(0.75, 6.25, size=N)},
                  columns=['basket_id', 'fruit', 'count', 'weight'])
df

,basket_id,fruit,count,weight
0,0,apple,11,2.901103
1,1,orange,5,2.580477
2,2,apple,12,4.040698
3,3,apple,6,1.777038
4,4,apple,5,4.450158
5,5,orange,12,5.929916
6,6,apple,10,2.115351
7,7,apple,11,5.968846


这里，df['fruit']是一个Python字符串对象的数组。我们可以通过调用：将其转换为分类变量：

In [147]:
fruit_cat = df['fruit'].astype('category')
fruit_cat

0     apple
1    orange
2     apple
3     apple
4     apple
5    orange
6     apple
7     apple
Name: fruit, dtype: category
Categories (2, object): ['apple', 'orange']

现在fruit_cat的值是pandas.Categorical的一个实例，您可以通过.array属性访问它：

In [148]:
c = fruit_cat.array
c 

['apple', 'orange', 'apple', 'apple', 'apple', 'orange', 'apple', 'apple']
Categories (2, object): ['apple', 'orange']

In [149]:
c.codes

array([0, 1, 0, 0, 0, 1, 0, 0], dtype=int8)

In [150]:
c.categories

Index(['apple', 'orange'], dtype='object')

一个有用的技巧是获得代码和类别之间映射的方法是：

In [151]:
dict(enumerate(c.categories))

{0: 'apple', 1: 'orange'}

通过指定转换结果，可以将DataFrame列转换为分类变量：

In [152]:
df['fruit'] = df['fruit'].astype('category')
df['fruit']

0     apple
1    orange
2     apple
3     apple
4     apple
5    orange
6     apple
7     apple
Name: fruit, dtype: category
Categories (2, object): ['apple', 'orange']

您还可以直接从其他类型的Python序列创建pandas.Categorical：

In [153]:
my_category = pd.Categorical(['foo', 'bar', 'baz', 'foo', 'bar'])
my_category

['foo', 'bar', 'baz', 'foo', 'bar']
Categories (3, object): ['bar', 'baz', 'foo']

如果您已经从其他来源获取了分类编码数据，您可以使用替代的from_codes构造函数：

In [154]:
categories = ['foo', 'bar', 'baz']
codes = [0, 1, 2, 0, 0, 1]
my_category = pd.Categorical.from_codes(codes, categories)
my_category

['foo', 'bar', 'baz', 'foo', 'foo', 'bar']
Categories (3, object): ['foo', 'bar', 'baz']

除非明确指定，分类转换不假设类别有特定的排序。因此，类别数组可能根据输入数据的排序而处于不同的顺序。当使用from_codes或任何其他构造函数时，您可以指示类别具有有意义的排序：

In [155]:
ordered_cat = pd.Categorical.from_codes(codes, categories,
                                       ordered=True)
ordered_cat

['foo', 'bar', 'baz', 'foo', 'foo', 'bar']
Categories (3, object): ['foo' < 'bar' < 'baz']

输出[foo < bar < baz]表示在排序中“foo”先于“bar”，依此类推。无序的分类实例可以通过as_ordered方法变为有序：

In [156]:
my_category.as_ordered()

['foo', 'bar', 'baz', 'foo', 'foo', 'bar']
Categories (3, object): ['foo' < 'bar' < 'baz']

最后说明一下，分类数据不必是字符串，尽管我只展示了字符串的例子。分类数组可以包含任何不可变值类型。

### 对分类数据的计算

在pandas中使用分类数据与非编码版本（如字符串数组）相比通常表现相同。pandas的某些部分，如groupby函数，在使用分类数据时性能更好。还有一些函数可以利用有序标志。

让我们考虑一些随机的数值数据并使用pandas.qcut分箱函数。这返回pandas.Categorical；我们在书中前面使用过pandas.cut，但忽略了分类器如何工作的细节：

In [157]:
rng = np.random.default_rng(seed=12345)
draws = rng.standard_normal(1000)
draws

array([-1.42382504e+00,  1.26372846e+00, -8.70661738e-01, -2.59173235e-01,
       -7.53433070e-02, -7.40884652e-01, -1.36779270e+00,  6.48892802e-01,
        3.61058113e-01, -1.95286306e+00,  2.34740965e+00,  9.68496906e-01,
       -7.59387180e-01,  9.02198274e-01, -4.66953173e-01, -6.06895187e-02,
        7.88844345e-01, -1.25666813e+00,  5.75857514e-01,  1.39897899e+00,
        1.32229806e+00, -2.99698515e-01,  9.02919341e-01, -1.62158273e+00,
       -1.58189261e-01,  4.49483932e-01, -1.34360107e+00, -8.16875907e-02,
        1.72473993e+00,  2.61815943e+00,  7.77361344e-01,  8.28633196e-01,
       -9.58988313e-01, -1.20938829e+00, -1.41229201e+00,  5.41546830e-01,
        7.51939396e-01, -6.58760320e-01, -1.22867499e+00,  2.57557768e-01,
        3.12902918e-01, -1.30811690e-01,  1.26998312e+00, -9.29624577e-02,
       -6.61508890e-02, -1.10821447e+00,  1.35956851e-01,  1.34707776e+00,
        6.11440210e-02,  7.09146003e-02,  4.33654537e-01,  2.77483660e-01,
        5.30252387e-01,  

让我们计算一下这个数据的分位数分组，并提取一些统计数据：

In [159]:
bins = pd.qcut(draws, 4, precision=3)
bins

[(-3.121, -0.675], (0.687, 3.211], (-3.121, -0.675], (-0.675, 0.0134], (-0.675, 0.0134], ..., (0.0134, 0.687], (0.0134, 0.687], (-0.675, 0.0134], (0.0134, 0.687], (-0.675, 0.0134]]
Length: 1000
Categories (4, interval[float64, right]): [(-3.121, -0.675] < (-0.675, 0.0134] < (0.0134, 0.687] < (0.687, 3.211]]

虽然精确的样本分位数对于生成报告可能不如四分位数的名称有用，但我们可以通过qcut的labels参数来实现这一点：

In [160]:
bins = pd.qcut(draws, 4, precision=3, labels=["Q1", "Q2", "Q3", "Q4"])
bins

['Q1', 'Q4', 'Q1', 'Q2', 'Q2', ..., 'Q3', 'Q3', 'Q2', 'Q3', 'Q2']
Length: 1000
Categories (4, object): ['Q1' < 'Q2' < 'Q3' < 'Q4']

In [161]:
bins.codes

array([0, 3, 0, 1, 1, 0, 0, 2, 2, 0, 3, 3, 0, 3, 1, 1, 3, 0, 2, 3, 3, 1,
       3, 0, 1, 2, 0, 1, 3, 3, 3, 3, 0, 0, 0, 2, 3, 1, 0, 2, 2, 1, 3, 1,
       1, 0, 2, 3, 2, 2, 2, 2, 2, 2, 2, 0, 2, 0, 2, 0, 1, 2, 1, 2, 0, 0,
       3, 0, 2, 0, 0, 0, 1, 2, 0, 1, 3, 1, 0, 2, 1, 0, 0, 3, 1, 1, 0, 3,
       3, 3, 1, 2, 3, 3, 2, 1, 1, 0, 0, 2, 1, 0, 1, 3, 1, 0, 1, 3, 2, 1,
       3, 3, 1, 3, 2, 3, 3, 3, 2, 1, 0, 1, 3, 2, 2, 2, 2, 2, 0, 0, 1, 0,
       3, 1, 1, 0, 2, 1, 3, 3, 0, 2, 0, 1, 3, 3, 2, 1, 0, 0, 2, 3, 0, 2,
       2, 3, 0, 1, 2, 0, 0, 0, 1, 2, 1, 2, 0, 0, 3, 1, 1, 3, 3, 3, 2, 1,
       2, 2, 3, 2, 2, 1, 3, 0, 2, 3, 0, 3, 1, 1, 0, 3, 2, 1, 2, 2, 3, 2,
       0, 1, 0, 0, 3, 2, 1, 1, 3, 2, 3, 2, 1, 3, 0, 0, 1, 0, 1, 1, 3, 2,
       2, 0, 0, 3, 0, 3, 0, 2, 0, 2, 0, 1, 3, 1, 3, 1, 0, 2, 1, 3, 3, 3,
       2, 3, 3, 2, 0, 0, 2, 3, 3, 0, 2, 2, 0, 2, 0, 2, 3, 0, 2, 0, 0, 3,
       0, 1, 1, 1, 2, 2, 3, 3, 1, 3, 3, 0, 2, 3, 3, 2, 0, 1, 0, 1, 3, 2,
       3, 1, 0, 0, 2, 0, 1, 3, 0, 0, 1, 1, 1, 0, 2,

标记的箱分类不包含数据中箱边的信息，因此我们可以使用groupby来提取一些摘要统计数据：

In [162]:
bins = pd.Series(bins, name="quartile")
results = pd.Series(draws).groupby(bins).agg(['count', 'min', 'max']).reset_index()
results

/var/folders/c5/vh03t8zn4797kc18lgrjtcbr0000gn/T/ipykernel_4990/2268927521.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  results = pd.Series(draws).groupby(bins).agg(['count', 'min', 'max']).reset_index()


,quartile,count,min,max
0,Q1,250,-3.119609,-0.678494
1,Q2,250,-0.673305,0.008009
2,Q3,250,0.018753,0.686183
3,Q4,250,0.688282,3.211418


结果中的“quartile”列保留了来自分箱的原始分类信息，包括排序：

In [163]:
results['quartile']

0    Q1
1    Q2
2    Q3
3    Q4
Name: quartile, dtype: category
Categories (4, object): ['Q1' < 'Q2' < 'Q3' < 'Q4']

#### 使用分类变量性能更好

在本节开头，我说过分类类型可以提高性能和内存使用效率，让我们来看一些例子。考虑一个包含1亿个元素且类别数量较少的Series：

In [164]:
N = 10_000_000
labels = pd.Series(['foo', 'bar', 'baz', 'qux'] * (N // 4))
categories = labels.astype('category')
categories

0          foo
1          bar
2          baz
3          qux
4          foo
          ... 
9999995    qux
9999996    foo
9999997    bar
9999998    baz
9999999    qux
Length: 10000000, dtype: category
Categories (4, object): ['bar', 'baz', 'foo', 'qux']

现在我们注意到标签使用的内存比类别要多得多：

In [165]:
labels.memory_usage(deep=True)

520000132

In [166]:
categories.memory_usage(deep=True)

10000512

当然，这种转换不是免费的，但是是一次性的费用：

In [167]:
%time _ = labels.astype('category')

CPU times: user 236 ms, sys: 36.8 ms, total: 272 ms
Wall time: 272 ms


使用分类变量时，GroupBy操作可能会快得多，因为底层算法使用的是基于整数的代码数组而不是字符串数组。这里我们比较了value_counts()的性能，该函数内部使用了GroupBy机制：

In [168]:
%timeit labels.value_counts()

244 ms ± 15.6 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [169]:
%timeit categories.value_counts()

23.3 ms ± 34.7 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


### 分类方法

包含分类数据的系列有几个特殊方法，类似于Series.str专门的字符串方法。这也提供了对类别和代码的便捷访问。考虑下面的Series：

In [170]:
s = pd.Series(['a', 'b', 'c', 'd'] * 2)
cat_s = s.astype('category')
cat_s

0    a
1    b
2    c
3    d
4    a
5    b
6    c
7    d
dtype: category
Categories (4, object): ['a', 'b', 'c', 'd']

In [171]:
cat_s.cat.codes

0    0
1    1
2    2
3    3
4    0
5    1
6    2
7    3
dtype: int8

In [172]:
cat_s.cat.categories

Index(['a', 'b', 'c', 'd'], dtype='object')

假设我们知道实际类别集合超出了数据中观察到的四个值。我们可以使用set_categories方法来更改它们：


In [173]:
actual_categories = ['a', 'b', 'c', 'd', 'e']
cat_s2 = cat_s.cat.set_categories(actual_categories)
cat_s2

0    a
1    b
2    c
3    d
4    a
5    b
6    c
7    d
dtype: category
Categories (5, object): ['a', 'b', 'c', 'd', 'e']

虽然数据似乎没有变化，但使用这些新类别的操作将反映它们。例如，如果存在类别，则value_counts会尊重这些类别：

In [174]:
cat_s.value_counts()

a    2
b    2
c    2
d    2
Name: count, dtype: int64

In [175]:
cat_s2.value_counts()

a    2
b    2
c    2
d    2
e    0
Name: count, dtype: int64

在大型数据集中，分类变量通常被用作节省内存和提高性能的便捷工具。当你过滤掉一个大型DataFrame或Series后，许多分类可能不会出现在数据中。为了解决这个问题，我们可以使用remove_unused_categories方法来修剪未观察到的分类：

In [176]:
cat_s3 = cat_s[cat_s.isin(['a', 'b'])]
cat_s3

0    a
1    b
4    a
5    b
dtype: category
Categories (4, object): ['a', 'b', 'c', 'd']

In [177]:
cat_s3.cat.remove_unused_categories()

0    a
1    b
4    a
5    b
dtype: category
Categories (2, object): ['a', 'b']

**pandas中Series的类别方法:**

| 方法 | 描述 |
|------|-----|
| add_categories | 在现有类别末尾添加新的（未使用的）类别 |
| as_ordered | 使类别按顺序排列 |
| as_unordered | 使分类无序 |
| remove_categories | 移除类别，并将任何被移除的值设置为null |
| remove_unused_categories | 删除数据中未出现的任何类别值 |
| rename_categories | 用指定的一组新类别名称替换类别；不能更改类别的数量。 |
| reorder_categories | 行为类似于rename_categories，但也可以更改结果以具有有序类别 |
| set_categories | 用指定的新类别集替换类别；可以添加或删除类别 |

#### 创建虚拟变量以建模

在使用统计或机器学习工具时，您通常会将要处理的分类数据转换为虚拟变量，也称为独热编码。这涉及到创建一个DataFrame，其中每个不同的类别都有一个列；这些列包含给定类别的出现次数（用1表示）以及其他情况下的0。
考虑前面的例子：

In [178]:
cat_s = pd.Series(['a', 'b', 'c', 'd'] * 2).astype('category')
cat_s

0    a
1    b
2    c
3    d
4    a
5    b
6    c
7    d
dtype: category
Categories (4, object): ['a', 'b', 'c', 'd']

如本章前面提到的，pandas.get_dummies函数将这个一维分类数据转换为一个包含虚拟变量的DataFrame：

In [180]:
pd.get_dummies(cat_s, dtype=float)

,a,b,c,d
0,1.0,0.0,0.0,0.0
1,0.0,1.0,0.0,0.0
2,0.0,0.0,1.0,0.0
3,0.0,0.0,0.0,1.0
4,1.0,0.0,0.0,0.0
5,0.0,1.0,0.0,0.0
6,0.0,0.0,1.0,0.0
7,0.0,0.0,0.0,1.0
